In [1]:
!pip install autofe-grass

<h3>Imports</h3>

In [9]:
from sklearn.datasets import fetch_openml
from autofe.stats import GroupAggregationFeatures, StatisticalFeatureGenerator
import pandas as pd
import numpy as np

import warnings
warnings.simplefilter('ignore')

<h3>Dataset loads</h3>

In [10]:
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame

print(f"Исходный размер данных: {df.shape}")
df.info()

Исходный размер данных: (1309, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   object  
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   object  
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    object  
 10  embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(3), float64(3), int64(3), object(5)
memory usage: 116.8+ KB


In [11]:
df.drop(columns=['survived'], inplace=True)

df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['pclass'] = df['pclass'].astype(int)
df['sibsp'] = df['sibsp'].astype(int)
df['parch'] = df['parch'].astype(int)

numeric_cols = ['age', 'fare', 'sibsp', 'parch']
group_cols = ['pclass']  # Группируем по классу билета

<h3>Basic pipline usage</h3>

In [12]:
group_feats1 = GroupAggregationFeatures(
        numeric_cols=numeric_cols,
        group_cols=group_cols,
        aggs=['mean', 'std'],  # только среднее и стандартное отклонение
        add_deviations=True,   # добавляем diff и ratio
        add_rank=False
)

# fit only
group_feats1.fit(df)
df_with_meta = group_feats1.transform(df, meta_usage=True)

df_with_meta.head()

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,...,std_sibsp,std_parch,diff_mean_age,ratio_mean_age,diff_mean_fare,ratio_mean_fare,diff_mean_sibsp,ratio_mean_sibsp,diff_mean_parch,ratio_mean_parch
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,...,0.609064,0.715602,-8.812436,0.766943,123.828508,2.415038,-0.436533,0.00000,-0.365325,0.000000
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,...,0.609064,0.715602,-36.895736,0.024243,64.041008,1.731822,0.563467,2.29078,1.634675,5.474576
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,...,0.609064,0.715602,-35.812436,0.052893,64.041008,1.731822,0.563467,2.29078,1.634675,5.474576
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,...,0.609064,0.715602,-7.812436,0.793390,64.041008,1.731822,0.563467,2.29078,1.634675,5.474576
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,...,0.609064,0.715602,-12.812436,0.661158,64.041008,1.731822,0.563467,2.29078,1.634675,5.474576


In [13]:
df_with_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   pclass            1309 non-null   int64   
 1   name              1309 non-null   object  
 2   sex               1309 non-null   category
 3   age               1309 non-null   float64 
 4   sibsp             1309 non-null   int64   
 5   parch             1309 non-null   int64   
 6   ticket            1309 non-null   object  
 7   fare              1309 non-null   float64 
 8   cabin             295 non-null    object  
 9   embarked          1307 non-null   category
 10  boat              486 non-null    object  
 11  body              121 non-null    float64 
 12  home.dest         745 non-null    object  
 13  mean_age          1309 non-null   float64 
 14  mean_fare         1309 non-null   float64 
 15  mean_sibsp        1309 non-null   float64 
 16  mean_parch        1309 n

<h3>Explanable auto feature engineering</h3>



In [14]:
df_with_meta.metadata

{'mean_age': {'operation': 'mean',
  'base_column': 'age',
  'type': 'aggregation',
  'group_cols': ['pclass'],
  'transformer': 'GroupAggregationFeatures'},
 'mean_fare': {'operation': 'mean',
  'base_column': 'fare',
  'type': 'aggregation',
  'group_cols': ['pclass'],
  'transformer': 'GroupAggregationFeatures'},
 'mean_sibsp': {'operation': 'mean',
  'base_column': 'sibsp',
  'type': 'aggregation',
  'group_cols': ['pclass'],
  'transformer': 'GroupAggregationFeatures'},
 'mean_parch': {'operation': 'mean',
  'base_column': 'parch',
  'type': 'aggregation',
  'group_cols': ['pclass'],
  'transformer': 'GroupAggregationFeatures'},
 'std_age': {'operation': 'std',
  'base_column': 'age',
  'type': 'aggregation',
  'group_cols': ['pclass'],
  'transformer': 'GroupAggregationFeatures'},
 'std_fare': {'operation': 'std',
  'base_column': 'fare',
  'type': 'aggregation',
  'group_cols': ['pclass'],
  'transformer': 'GroupAggregationFeatures'},
 'std_sibsp': {'operation': 'std',
  'base_c

In [15]:
df_with_meta.metadata['ratio_mean_sibsp']

{'operation': 'ratio',
 'base_aggregation': 'mean',
 'base_column': 'sibsp',
 'type': 'deviation',
 'formula': 'sibsp / mean(sibsp)',
 'transformer': 'GroupAggregationFeatures'}